# Module 41: Building a Diffusion Model — From Noise to Data

Build a complete diffusion model from scratch: noise schedules, UNet architecture, DDPM and DDIM sampling — all on 2D distributions for clear visualization.

| Input | Output |
|-------|--------|
| `Pure Gaussian noise (2D)` | `Swiss roll distribution` |
| `1000 DDPM steps` | `50 DDIM steps (20x faster)` |

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch {torch.__version__}")
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. What Are Diffusion Models?

Diffusion models generate data by **reversing a gradual noising process**:

```
Forward:  x_0 (data) --> x_1 --> ... --> x_T (pure noise)   [fixed]
Reverse:  x_T (noise) --> ... --> x_1 --> x_0 (generated)   [learned]
```

The model learns to predict the noise added at each step, effectively learning the data distribution.

## 2. Noise Schedules

The noise schedule `{beta_1, ..., beta_T}` controls how fast data is destroyed. We implement **linear** and **cosine** schedules.

In [ ]:
def linear_beta_schedule(num_timesteps, beta_start=1e-4, beta_end=0.02):
    """Linear interpolation from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, num_timesteps)


def cosine_beta_schedule(num_timesteps, s=0.008):
    """Cosine schedule from Improved DDPM (Nichol & Dhariwal 2021)."""
    steps = torch.linspace(0, num_timesteps, num_timesteps + 1)
    alphas_cumprod = torch.cos((steps / num_timesteps + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)


# Compare schedules
linear_betas = linear_beta_schedule(1000)
cosine_betas = cosine_beta_schedule(1000)

print(f"Linear beta range: [{linear_betas[0]:.6f}, {linear_betas[-1]:.6f}]")
print(f"Cosine beta range: [{cosine_betas[0]:.6f}, {cosine_betas[-1]:.6f}]")

## 3. Comparing alpha_bar (Cumulative Signal Retention)

`alpha_bar_t` tells us how much of the original signal remains at timestep `t`.

In [ ]:
linear_alphas = 1.0 - linear_betas
linear_alphas_cumprod = torch.cumprod(linear_alphas, dim=0)

cosine_alphas = 1.0 - cosine_betas
cosine_alphas_cumprod = torch.cumprod(cosine_alphas, dim=0)

print(f"{'Timestep':>10} {'Linear a_bar':>14} {'Cosine a_bar':>14}")
print("-" * 40)
for t in [0, 100, 250, 500, 750, 900, 999]:
    print(f"{t:>10d} {linear_alphas_cumprod[t]:>14.6f} {cosine_alphas_cumprod[t]:>14.6f}")

print(f"\nCosine preserves signal longer -- better for training!")

## 4. DiffusionSchedule Class

Precomputes all derived quantities needed for training and sampling.

In [ ]:
class DiffusionSchedule:
    """Precompute and store all quantities for DDPM training/sampling."""

    def __init__(self, num_timesteps=1000, schedule_type="cosine"):
        self.num_timesteps = num_timesteps

        if schedule_type == "linear":
            self.betas = linear_beta_schedule(num_timesteps)
        else:
            self.betas = cosine_beta_schedule(num_timesteps)

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_0, t, noise):
        """Forward process: x_t = sqrt(a_bar_t) * x_0 + sqrt(1-a_bar_t) * noise."""
        sqrt_alpha = self.sqrt_alphas_cumprod[t]
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t]
        while sqrt_alpha.dim() < x_0.dim():
            sqrt_alpha = sqrt_alpha.unsqueeze(-1)
            sqrt_one_minus = sqrt_one_minus.unsqueeze(-1)
        return sqrt_alpha * x_0 + sqrt_one_minus * noise

    def to(self, device):
        for attr in ["betas", "alphas", "alphas_cumprod", "alphas_cumprod_prev",
                     "sqrt_alphas_cumprod", "sqrt_one_minus_alphas_cumprod",
                     "sqrt_recip_alphas", "posterior_variance"]:
            setattr(self, attr, getattr(self, attr).to(device))
        return self


schedule = DiffusionSchedule(1000, "cosine")
print(f"Schedule created: T={schedule.num_timesteps}")
print(f"alpha_bar range: [{schedule.alphas_cumprod[-1]:.6f}, {schedule.alphas_cumprod[0]:.6f}]")

## 5. Forward Process Demo

The forward process `q(x_t | x_0)` adds noise to data. We can jump to any timestep directly.

In [ ]:
# Create simple 2D data
torch.manual_seed(42)
data_2d = torch.randn(100, 2) * 0.5 + torch.tensor([1.0, 1.0])

print("Forward process: adding noise to data at different timesteps")
print(f"Original data mean: ({data_2d[:, 0].mean():.3f}, {data_2d[:, 1].mean():.3f})")
print(f"Original data std:  ({data_2d[:, 0].std():.3f}, {data_2d[:, 1].std():.3f})")

noise = torch.randn_like(data_2d)
for t_val in [0, 100, 250, 500, 750, 999]:
    t = torch.full((100,), t_val, dtype=torch.long)
    x_t = schedule.q_sample(data_2d, t, noise)
    snr = schedule.sqrt_alphas_cumprod[t_val] / schedule.sqrt_one_minus_alphas_cumprod[t_val]
    print(f"  t={t_val:>4d}: mean=({x_t[:, 0].mean():.3f}, {x_t[:, 1].mean():.3f}), "
          f"std=({x_t[:, 0].std():.3f}, {x_t[:, 1].std():.3f}), SNR={snr:.4f}")

## 6. 2D Data Generators

We train on 2D distributions for intuitive visualization.

In [ ]:
def make_swiss_roll(n_samples=2000):
    t = 1.5 * math.pi * (1 + 2 * torch.rand(n_samples))
    x = t * torch.cos(t)
    y = t * torch.sin(t)
    data = torch.stack([x, y], dim=-1)
    return (data - data.mean(0)) / data.std()


def make_moons(n_samples=2000):
    n = n_samples // 2
    theta1 = torch.linspace(0, math.pi, n)
    x1, y1 = torch.cos(theta1), torch.sin(theta1)
    theta2 = torch.linspace(0, math.pi, n_samples - n)
    x2, y2 = 1 - torch.cos(theta2), 1 - torch.sin(theta2) - 0.5
    x = torch.cat([x1, x2]) + torch.randn(n_samples) * 0.05
    y = torch.cat([y1, y2]) + torch.randn(n_samples) * 0.05
    data = torch.stack([x, y], dim=-1)
    return (data - data.mean(0)) / data.std()


def make_circles(n_samples=2000):
    n = n_samples // 2
    t1 = torch.linspace(0, 2 * math.pi, n + 1)[:-1]
    t2 = torch.linspace(0, 2 * math.pi, (n_samples - n) + 1)[:-1]
    x = torch.cat([torch.cos(t1) + torch.randn(n)*0.05, 0.5*torch.cos(t2) + torch.randn(n_samples-n)*0.05])
    y = torch.cat([torch.sin(t1) + torch.randn(n)*0.05, 0.5*torch.sin(t2) + torch.randn(n_samples-n)*0.05])
    data = torch.stack([x, y], dim=-1)
    return (data - data.mean(0)) / data.std()


torch.manual_seed(42)
swiss_roll = make_swiss_roll(2000)
moons = make_moons(2000)
circles = make_circles(2000)

for name, data in [("Swiss Roll", swiss_roll), ("Moons", moons), ("Circles", circles)]:
    print(f"{name}: shape={data.shape}, mean=({data[:, 0].mean():.3f}, {data[:, 1].mean():.3f}), "
          f"std=({data[:, 0].std():.3f}, {data[:, 1].std():.3f})")

## 7. Visualize Forward Process on Swiss Roll

Watch data become noise as `t` increases.

In [ ]:
print("Forward process on Swiss Roll:")
print(f"{'t':>6} {'Signal':>8} {'Noise':>8} {'Sample (0)':>20} {'Sample (1)':>20}")
print("-" * 65)

noise = torch.randn_like(swiss_roll[:5])
for t_val in [0, 50, 100, 250, 500, 750, 999]:
    t = torch.full((5,), t_val, dtype=torch.long)
    x_t = schedule.q_sample(swiss_roll[:5], t, noise)
    sig = schedule.sqrt_alphas_cumprod[t_val].item()
    noi = schedule.sqrt_one_minus_alphas_cumprod[t_val].item()
    print(f"{t_val:>6d} {sig:>8.4f} {noi:>8.4f} ({x_t[0,0]:>7.3f}, {x_t[0,1]:>7.3f})  ({x_t[1,0]:>7.3f}, {x_t[1,1]:>7.3f})")

## 8. Sinusoidal Time Embedding

Timesteps are encoded using sinusoidal positional encoding, giving the network a smooth representation of noise level.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Encode timesteps with sinusoidal positional encoding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


# Demo
embed = SinusoidalTimeEmbedding(64)
t_test = torch.tensor([0, 100, 500, 999])
emb = embed(t_test)
print(f"Embedding shape: {emb.shape}")

print("\nCosine similarity (nearby t should be more similar):")
for i in range(len(t_test)):
    for j in range(i+1, len(t_test)):
        sim = F.cosine_similarity(emb[i:i+1], emb[j:j+1]).item()
        print(f"  sim(t={t_test[i]}, t={t_test[j]}): {sim:.4f}")

## 9. ResBlock with Time Conditioning

Each residual block receives the time embedding, adapting its behavior based on noise level.

In [ ]:
class ResBlock(nn.Module):
    """Residual block with additive time conditioning."""
    def __init__(self, dim, time_dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim
        self.net = nn.Sequential(nn.Linear(dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, dim))
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, dim))

    def forward(self, x, t_emb):
        return x + self.net(x) + self.time_mlp(t_emb)


# Test
block = ResBlock(128, 64)
x = torch.randn(4, 128)
t_emb = torch.randn(4, 64)
out = block(x, t_emb)
print(f"ResBlock: {x.shape} -> {out.shape}")
print(f"Residual: output = x + net(x) + time_mlp(t_emb)")

## 10. Down and Up Blocks

DownBlocks reduce dimension and save skip connections. UpBlocks restore dimension using skips.

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, dim_in, dim_out, time_dim):
        super().__init__()
        self.res1 = ResBlock(dim_in, time_dim)
        self.res2 = ResBlock(dim_in, time_dim)
        self.downsample = nn.Linear(dim_in, dim_out)

    def forward(self, x, t_emb):
        x = self.res1(x, t_emb)
        x = self.res2(x, t_emb)
        skip = x
        x = self.downsample(x)
        return x, skip


class UpBlock(nn.Module):
    def __init__(self, dim_in, dim_out, time_dim):
        super().__init__()
        self.upsample = nn.Linear(dim_in, dim_out)
        self.res1 = ResBlock(dim_out * 2, time_dim, hidden_dim=dim_out)
        self.proj = nn.Linear(dim_out * 2, dim_out)
        self.res2 = ResBlock(dim_out, time_dim)

    def forward(self, x, skip, t_emb):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=-1)
        x = self.res1(x, t_emb)
        x = self.proj(x)
        x = self.res2(x, t_emb)
        return x


class MidBlock(nn.Module):
    def __init__(self, dim, time_dim):
        super().__init__()
        self.res1 = ResBlock(dim, time_dim)
        self.res2 = ResBlock(dim, time_dim)

    def forward(self, x, t_emb):
        return self.res2(self.res1(x, t_emb), t_emb)


# Test blocks
down = DownBlock(256, 128, 64)
up = UpBlock(128, 256, 64)
x = torch.randn(4, 256)
t_emb = torch.randn(4, 64)
x_down, skip = down(x, t_emb)
x_up = up(x_down, skip, t_emb)
print(f"Down: {x.shape} -> {x_down.shape} (skip: {skip.shape})")
print(f"Up:   {x_down.shape} + skip -> {x_up.shape}")

## 11. Complete PointUNet

The full UNet for 2D point diffusion: input (B, 2) noisy points + timestep, output (B, 2) predicted noise.

In [ ]:
class PointUNet(nn.Module):
    """UNet for denoising 2D point distributions."""
    def __init__(self, input_dim=2, hidden_dims=(256, 128, 64), time_dim=128):
        super().__init__()
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 2), nn.SiLU(), nn.Linear(time_dim * 2, time_dim),
        )
        self.input_proj = nn.Linear(input_dim, hidden_dims[0])

        self.downs = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            self.downs.append(DownBlock(hidden_dims[i], hidden_dims[i+1], time_dim))

        self.mid = MidBlock(hidden_dims[-1], time_dim)

        self.ups = nn.ModuleList()
        for i in range(len(hidden_dims) - 2, -1, -1):
            self.ups.append(UpBlock(hidden_dims[i+1], hidden_dims[i], time_dim))

        self.output_proj = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dims[0], input_dim))

    def forward(self, x, t):
        t_emb = self.time_mlp(self.time_embed(t))
        x = self.input_proj(x)

        skips = []
        for down in self.downs:
            x, skip = down(x, t_emb)
            skips.append(skip)

        x = self.mid(x, t_emb)

        for up in self.ups:
            x = up(x, skips.pop(), t_emb)

        return self.output_proj(x)


model = PointUNet(input_dim=2, hidden_dims=(256, 128, 64), time_dim=128)
num_params = sum(p.numel() for p in model.parameters())
print(f"PointUNet parameters: {num_params:,}")

# Test forward pass
x = torch.randn(32, 2)
t = torch.randint(0, 1000, (32,))
pred = model(x, t)
print(f"Input: {x.shape}, Timestep: {t.shape} -> Output: {pred.shape}")

## 12. Gradient Flow Check

Verify gradients flow correctly through all layers.

In [ ]:
x = torch.randn(8, 2)
t = torch.randint(0, 1000, (8,))
target = torch.randn(8, 2)

pred = model(x, t)
loss = F.mse_loss(pred, target)
loss.backward()

num_with_grad = sum(1 for p in model.parameters() if p.grad is not None)
num_total = sum(1 for _ in model.parameters())
print(f"Parameters with gradients: {num_with_grad}/{num_total}")
print(f"Loss: {loss.item():.6f}")
print("All parameters receive gradients!")

model.zero_grad()

## 13. DDPM Training Loop

The training algorithm:
1. Sample `x_0` from data
2. Sample random timestep `t`
3. Sample noise `epsilon`
4. Forward process: `x_t = sqrt(a_bar_t) * x_0 + sqrt(1-a_bar_t) * epsilon`
5. Predict noise: `eps_hat = model(x_t, t)`
6. Loss: `MSE(eps_hat, epsilon)`

In [ ]:
def train_diffusion(model, schedule, data, num_epochs=100, batch_size=256, lr=1e-3):
    """DDPM training loop."""
    dev = next(model.parameters()).device
    data = data.to(dev)
    loader = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched_lr = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr*0.01)

    losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        n = 0
        for (x_0,) in loader:
            t = torch.randint(0, schedule.num_timesteps, (x_0.shape[0],), device=dev)
            noise = torch.randn_like(x_0)
            x_t = schedule.q_sample(x_0, t, noise)

            pred = model(x_t, t)
            loss = F.mse_loss(pred, noise)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n += 1

        sched_lr.step()
        avg = epoch_loss / n
        losses.append(avg)
        if (epoch+1) % 20 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:>3d}/{num_epochs} | Loss: {avg:.6f}")

    return losses


print("Training function defined.")

## 14. Train on Swiss Roll

Let's train the diffusion model on the Swiss roll distribution.

In [ ]:
torch.manual_seed(42)

# Fresh model and schedule
schedule = DiffusionSchedule(1000, "cosine").to(device)
model = PointUNet(input_dim=2, hidden_dims=(256, 128, 64), time_dim=128).to(device)

print(f"Training on Swiss Roll ({swiss_roll.shape[0]} points)...")
t0 = time.time()
losses = train_diffusion(model, schedule, swiss_roll, num_epochs=100, batch_size=256, lr=1e-3)
print(f"\nTotal training time: {time.time()-t0:.1f}s")
print(f"Final loss: {losses[-1]:.6f} (started at {losses[0]:.6f})")

## 15. Training Loss Curve

In [ ]:
print("Training Loss Curve:")
print(f"{'Epoch':>6} {'Loss':>10}")
print("-" * 18)
for i, l in enumerate(losses):
    if (i+1) % 10 == 0 or i == 0:
        bar = '#' * int(50 * (1 - l / max(losses)))
        print(f"{i+1:>6d} {l:>10.6f} |{bar}")

## 16. DDPM Sampling

Generate samples by iteratively denoising from pure noise (1000 steps).

In [ ]:
@torch.no_grad()
def ddpm_sample(model, schedule, num_samples, data_dim=2, device="cpu"):
    """DDPM reverse process: iterative denoising from pure noise."""
    model.eval()
    x = torch.randn(num_samples, data_dim, device=device)

    for t in reversed(range(schedule.num_timesteps)):
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)

        alpha = schedule.alphas[t]
        alpha_bar = schedule.alphas_cumprod[t]
        beta = schedule.betas[t]

        mean = schedule.sqrt_recip_alphas[t] * (
            x - beta / schedule.sqrt_one_minus_alphas_cumprod[t] * pred_noise
        )

        if t > 0:
            x = mean + beta.sqrt() * torch.randn_like(x)
        else:
            x = mean

    model.train()
    return x


print("DDPM Sampling (1000 steps)...")
t0 = time.time()
ddpm_samples = ddpm_sample(model, schedule, 500, device=device).cpu()
ddpm_time = time.time() - t0

print(f"Time: {ddpm_time:.2f}s")
print(f"Generated {ddpm_samples.shape[0]} samples")
print(f"Mean: ({ddpm_samples[:, 0].mean():.4f}, {ddpm_samples[:, 1].mean():.4f})")
print(f"Std:  ({ddpm_samples[:, 0].std():.4f}, {ddpm_samples[:, 1].std():.4f})")
print(f"\nFirst 10 generated points:")
for i in range(10):
    print(f"  ({ddpm_samples[i, 0]:.4f}, {ddpm_samples[i, 1]:.4f})")

## 17. DDIM Sampling

DDIM uses fewer steps by skipping timesteps with a non-Markovian update rule.

In [ ]:
@torch.no_grad()
def ddim_sample(model, schedule, num_samples, data_dim=2, device="cpu", num_steps=50, eta=0.0):
    """DDIM sampling: fewer steps, optionally deterministic."""
    model.eval()
    step_size = max(schedule.num_timesteps // num_steps, 1)
    timesteps = list(range(0, schedule.num_timesteps, step_size))[::-1]

    x = torch.randn(num_samples, data_dim, device=device)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)

        alpha_bar_t = schedule.alphas_cumprod[t]
        alpha_bar_prev = schedule.alphas_cumprod[timesteps[i+1]] if i < len(timesteps)-1 else torch.tensor(1.0, device=device)

        pred_x0 = (x - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()
        pred_x0 = pred_x0.clamp(-5, 5)

        sigma = eta * ((1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev)).sqrt()
        direction = (1 - alpha_bar_prev - sigma**2).clamp(min=0).sqrt() * pred_noise

        x = alpha_bar_prev.sqrt() * pred_x0 + direction
        if eta > 0 and i < len(timesteps) - 1:
            x = x + sigma * torch.randn_like(x)

    model.train()
    return x


print("DDIM Sampling (50 steps, deterministic)...")
t0 = time.time()
ddim_samples = ddim_sample(model, schedule, 500, device=device, num_steps=50, eta=0.0).cpu()
ddim_time = time.time() - t0

print(f"Time: {ddim_time:.2f}s ({ddpm_time/max(ddim_time, 1e-6):.1f}x faster than DDPM)")
print(f"Mean: ({ddim_samples[:, 0].mean():.4f}, {ddim_samples[:, 1].mean():.4f})")
print(f"Std:  ({ddim_samples[:, 0].std():.4f}, {ddim_samples[:, 1].std():.4f})")
print(f"\nFirst 10 generated points:")
for i in range(10):
    print(f"  ({ddim_samples[i, 0]:.4f}, {ddim_samples[i, 1]:.4f})")

## 18. Compare DDPM vs DDIM Samples

In [ ]:
print("Sample Quality Comparison (vs real Swiss Roll data):")
print(f"{'Metric':<25} {'Real':>10} {'DDPM':>10} {'DDIM-50':>10}")
print("-" * 57)

for name, s in [("Mean X", 0), ("Mean Y", 1)]:
    print(f"{name:<25} {swiss_roll[:, s].mean():>10.4f} {ddpm_samples[:, s].mean():>10.4f} {ddim_samples[:, s].mean():>10.4f}")
for name, s in [("Std X", 0), ("Std Y", 1)]:
    print(f"{name:<25} {swiss_roll[:, s].std():>10.4f} {ddpm_samples[:, s].std():>10.4f} {ddim_samples[:, s].std():>10.4f}")

real_cov = torch.cov(swiss_roll.T)
ddpm_cov = torch.cov(ddpm_samples.T)
ddim_cov = torch.cov(ddim_samples.T)
print(f"{'Cov Frobenius diff':<25} {'0':>10} {(real_cov - ddpm_cov).norm():>10.4f} {(real_cov - ddim_cov).norm():>10.4f}")

## 19. DDIM Step Count Comparison

How does sample quality change with the number of DDIM steps?

In [ ]:
print(f"{'Steps':>6} {'Time (s)':>10} {'Mean X':>10} {'Std X':>10} {'Cov Diff':>10}")
print("-" * 48)

for steps in [10, 25, 50, 100, 200]:
    t0 = time.time()
    s = ddim_sample(model, schedule, 500, device=device, num_steps=steps, eta=0.0).cpu()
    elapsed = time.time() - t0
    cov_diff = (real_cov - torch.cov(s.T)).norm().item()
    print(f"{steps:>6d} {elapsed:>10.3f} {s[:, 0].mean():>10.4f} {s[:, 0].std():>10.4f} {cov_diff:>10.4f}")

## 20. DDIM Determinism

With `eta=0`, DDIM is deterministic: same noise produces same output.

In [ ]:
torch.manual_seed(99)
s1 = ddim_sample(model, schedule, 5, device=device, num_steps=50, eta=0.0).cpu()
torch.manual_seed(99)
s2 = ddim_sample(model, schedule, 5, device=device, num_steps=50, eta=0.0).cpu()

diff = (s1 - s2).abs().max().item()
print(f"Max difference: {diff:.2e}")
print(f"Deterministic: {'Yes' if diff < 1e-5 else 'No'}")

print("\nRun 1:")
for i in range(5):
    print(f"  ({s1[i, 0]:.6f}, {s1[i, 1]:.6f})")
print("Run 2:")
for i in range(5):
    print(f"  ({s2[i, 0]:.6f}, {s2[i, 1]:.6f})")

## 21. Train on Two Moons

In [ ]:
torch.manual_seed(42)
schedule2 = DiffusionSchedule(1000, "cosine").to(device)
model2 = PointUNet(input_dim=2, hidden_dims=(256, 128, 64), time_dim=128).to(device)

print("Training on Two Moons...")
losses2 = train_diffusion(model2, schedule2, moons, num_epochs=80, batch_size=256, lr=1e-3)
print(f"Final loss: {losses2[-1]:.6f}")

moon_samples = ddim_sample(model2, schedule2, 500, device=device, num_steps=50).cpu()
print(f"\nGenerated moons:")
print(f"  Mean: ({moon_samples[:, 0].mean():.4f}, {moon_samples[:, 1].mean():.4f})")
print(f"  Std:  ({moon_samples[:, 0].std():.4f}, {moon_samples[:, 1].std():.4f})")
print(f"  First 5 points:")
for i in range(5):
    print(f"    ({moon_samples[i, 0]:.4f}, {moon_samples[i, 1]:.4f})")

## 22. Train on Circles

In [ ]:
torch.manual_seed(42)
schedule3 = DiffusionSchedule(1000, "cosine").to(device)
model3 = PointUNet(input_dim=2, hidden_dims=(256, 128, 64), time_dim=128).to(device)

print("Training on Circles...")
losses3 = train_diffusion(model3, schedule3, circles, num_epochs=80, batch_size=256, lr=1e-3)
print(f"Final loss: {losses3[-1]:.6f}")

circle_samples = ddim_sample(model3, schedule3, 500, device=device, num_steps=50).cpu()
print(f"\nGenerated circles:")
print(f"  Mean: ({circle_samples[:, 0].mean():.4f}, {circle_samples[:, 1].mean():.4f})")
print(f"  Std:  ({circle_samples[:, 0].std():.4f}, {circle_samples[:, 1].std():.4f})")
print(f"  First 5 points:")
for i in range(5):
    print(f"    ({circle_samples[i, 0]:.4f}, {circle_samples[i, 1]:.4f})")

## 23. Simple 1D UNet

The simplest possible diffusion model — for building intuition.

In [ ]:
class SimpleUNet1D(nn.Module):
    """Minimal 1D denoiser: concat input + time_emb -> MLP."""
    def __init__(self, time_dim=32, hidden_dim=128):
        super().__init__()
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.net = nn.Sequential(
            nn.Linear(1 + time_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_embed(t)], dim=-1))


model_1d = SimpleUNet1D().to(device)
print(f"SimpleUNet1D params: {sum(p.numel() for p in model_1d.parameters()):,}")

x_1d = torch.randn(16, 1, device=device)
t_1d = torch.randint(0, 1000, (16,), device=device)
out_1d = model_1d(x_1d, t_1d)
print(f"Input: {x_1d.shape} -> Output: {out_1d.shape}")

## 24. Classifier-Free Guidance Concept

In classifier-free guidance, the model learns both conditional and unconditional denoising.
At sampling time, we amplify the conditional signal.

In [ ]:
# Conceptual demo: how guidance changes the prediction
print("Classifier-Free Guidance (conceptual):")
print()
print("During training:")
print("  10% of the time: drop condition -> unconditional prediction")
print("  90% of the time: use condition  -> conditional prediction")
print()
print("During sampling:")
print("  eps_guided = eps_unconditional + w * (eps_conditional - eps_unconditional)")
print()

# Simulate with random vectors
eps_uncond = torch.tensor([0.5, 0.3])  # Unconditional prediction
eps_cond = torch.tensor([0.8, -0.2])   # Conditional prediction

print(f"{'w':>5} {'eps_guided':>30}  {'magnitude':>10}")
print("-" * 50)
for w in [0.0, 1.0, 3.0, 5.0, 7.5, 10.0]:
    guided = eps_uncond + w * (eps_cond - eps_uncond)
    mag = guided.norm().item()
    print(f"{w:>5.1f} ({guided[0]:>8.3f}, {guided[1]:>8.3f})     {mag:>10.4f}")

print("\nHigher w = stronger guidance (sharper but less diverse)")

## 25. Linear vs Cosine Schedule Effect on Training

In [ ]:
torch.manual_seed(42)

# Train with linear schedule
sched_lin = DiffusionSchedule(1000, "linear").to(device)
model_lin = PointUNet(input_dim=2, hidden_dims=(128, 64), time_dim=64).to(device)
print("Training with LINEAR schedule...")
losses_lin = train_diffusion(model_lin, sched_lin, swiss_roll, num_epochs=60, batch_size=256, lr=1e-3)

# Train with cosine schedule
torch.manual_seed(42)
sched_cos = DiffusionSchedule(1000, "cosine").to(device)
model_cos = PointUNet(input_dim=2, hidden_dims=(128, 64), time_dim=64).to(device)
print("\nTraining with COSINE schedule...")
losses_cos = train_diffusion(model_cos, sched_cos, swiss_roll, num_epochs=60, batch_size=256, lr=1e-3)

print(f"\nFinal loss: Linear={losses_lin[-1]:.6f}, Cosine={losses_cos[-1]:.6f}")

## 26. Compare Samples from Linear vs Cosine

In [ ]:
samples_lin = ddim_sample(model_lin, sched_lin, 500, device=device, num_steps=50).cpu()
samples_cos = ddim_sample(model_cos, sched_cos, 500, device=device, num_steps=50).cpu()

print(f"{'Metric':<20} {'Real':>10} {'Linear':>10} {'Cosine':>10}")
print("-" * 52)
print(f"{'Mean X':<20} {swiss_roll[:, 0].mean():>10.4f} {samples_lin[:, 0].mean():>10.4f} {samples_cos[:, 0].mean():>10.4f}")
print(f"{'Mean Y':<20} {swiss_roll[:, 1].mean():>10.4f} {samples_lin[:, 1].mean():>10.4f} {samples_cos[:, 1].mean():>10.4f}")
print(f"{'Std X':<20} {swiss_roll[:, 0].std():>10.4f} {samples_lin[:, 0].std():>10.4f} {samples_cos[:, 0].std():>10.4f}")
print(f"{'Std Y':<20} {swiss_roll[:, 1].std():>10.4f} {samples_lin[:, 1].std():>10.4f} {samples_cos[:, 1].std():>10.4f}")

rc = torch.cov(swiss_roll.T)
diff_lin = (rc - torch.cov(samples_lin.T)).norm().item()
diff_cos = (rc - torch.cov(samples_cos.T)).norm().item()
print(f"{'Cov diff':<20} {'0':>10} {diff_lin:>10.4f} {diff_cos:>10.4f}")

## 27. Stochastic vs Deterministic DDIM

In [ ]:
print("DDIM with different eta values (Swiss Roll model):")
print(f"{'eta':>5} {'Std X':>10} {'Std Y':>10} {'Cov diff':>10}")
print("-" * 38)

for eta in [0.0, 0.25, 0.5, 0.75, 1.0]:
    s = ddim_sample(model, schedule, 500, device=device, num_steps=50, eta=eta).cpu()
    cd = (real_cov - torch.cov(s.T)).norm().item()
    print(f"{eta:>5.2f} {s[:, 0].std():>10.4f} {s[:, 1].std():>10.4f} {cd:>10.4f}")

## 28. Reverse Process Visualization

Watch noise become data through the reverse process.

In [ ]:
@torch.no_grad()
def visualize_reverse_process(model, schedule, num_samples=5, device="cpu"):
    """Show intermediate steps during reverse process."""
    model.eval()
    x = torch.randn(num_samples, 2, device=device)
    checkpoints = [999, 750, 500, 250, 100, 50, 0]

    print(f"{'Step':>6} {'Point 0':>20} {'Point 1':>20}")
    print("-" * 48)

    for t in reversed(range(schedule.num_timesteps)):
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        pred = model(x, t_batch)

        mean = schedule.sqrt_recip_alphas[t] * (
            x - schedule.betas[t] / schedule.sqrt_one_minus_alphas_cumprod[t] * pred
        )
        x = mean + (schedule.betas[t].sqrt() * torch.randn_like(x) if t > 0 else 0)

        if t in checkpoints:
            print(f"t={t:>4d} ({x[0, 0]:>8.4f}, {x[0, 1]:>8.4f})  ({x[1, 0]:>8.4f}, {x[1, 1]:>8.4f})")

    model.train()


torch.manual_seed(42)
print("Reverse process (noise -> Swiss Roll):")
visualize_reverse_process(model, schedule, num_samples=5, device=device)

## 29. Summary Statistics Across All Distributions

In [ ]:
print("Summary of all trained models:")
print(f"{'Distribution':<15} {'Train Loss':>12} {'Gen Mean X':>12} {'Gen Std X':>12}")
print("-" * 53)

for name, loss_list, samples in [
    ("Swiss Roll", losses, ddpm_samples),
    ("Two Moons", losses2, moon_samples),
    ("Circles", losses3, circle_samples),
]:
    print(f"{name:<15} {loss_list[-1]:>12.6f} {samples[:, 0].mean():>12.4f} {samples[:, 0].std():>12.4f}")

## 30. Exercise: Modify the Noise Schedule

**Task**: Create a custom noise schedule and compare its effect on sample quality.

Try:
1. Change `beta_end` in the linear schedule (e.g., 0.01 vs 0.05)
2. Change the offset `s` in the cosine schedule (e.g., 0.001 vs 0.02)
3. Create a "sigmoid" schedule where betas follow a sigmoid curve

Questions to answer:
- How does each schedule affect final training loss?
- How does it affect the quality of generated samples?
- Which schedule distributes information destruction most evenly?

In [ ]:
# Exercise: Implement a sigmoid schedule and compare

def sigmoid_beta_schedule(num_timesteps, beta_start=1e-4, beta_end=0.02):
    """TODO: Implement a sigmoid-shaped beta schedule.
    
    Hint: use torch.sigmoid on a linspace from -6 to 6,
    then scale to [beta_start, beta_end].
    """
    # Your code here
    betas = torch.linspace(-6, 6, num_timesteps)
    betas = torch.sigmoid(betas)
    betas = betas * (beta_end - beta_start) + beta_start
    return betas


# Compare alpha_bar curves
for name, betas in [
    ("Linear", linear_beta_schedule(1000)),
    ("Cosine", cosine_beta_schedule(1000)),
    ("Sigmoid", sigmoid_beta_schedule(1000)),
]:
    abar = torch.cumprod(1.0 - betas, dim=0)
    print(f"{name:<10} alpha_bar[0]={abar[0]:.6f}, alpha_bar[500]={abar[500]:.6f}, alpha_bar[999]={abar[-1]:.6f}")

## 31. Exercise Solution & Comparison

In [ ]:
# Train with sigmoid schedule for comparison
torch.manual_seed(42)

class SigmoidSchedule(DiffusionSchedule):
    def __init__(self, num_timesteps=1000):
        # Override parent to use sigmoid
        self.num_timesteps = num_timesteps
        self.betas = sigmoid_beta_schedule(num_timesteps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)


sched_sig = SigmoidSchedule(1000).to(device)
model_sig = PointUNet(input_dim=2, hidden_dims=(128, 64), time_dim=64).to(device)
print("Training with SIGMOID schedule...")
losses_sig = train_diffusion(model_sig, sched_sig, swiss_roll, num_epochs=60, batch_size=256, lr=1e-3)

samples_sig = ddim_sample(model_sig, sched_sig, 500, device=device, num_steps=50).cpu()

print(f"\nComparison (60 epochs on Swiss Roll):")
print(f"{'Schedule':<10} {'Final Loss':>12} {'Gen Std X':>12} {'Cov Diff':>12}")
print("-" * 48)
for name, ll, ss in [
    ("Linear", losses_lin, samples_lin),
    ("Cosine", losses_cos, samples_cos),
    ("Sigmoid", losses_sig, samples_sig),
]:
    cd = (rc - torch.cov(ss.T)).norm().item()
    print(f"{name:<10} {ll[-1]:>12.6f} {ss[:, 0].std():>12.4f} {cd:>12.4f}")

## 32. Key Takeaways

1. **Diffusion models learn by denoising** -- the model predicts the noise added at each step
2. **Closed-form forward process** -- `q(x_t | x_0)` lets us jump to any timestep directly
3. **Cosine > Linear schedule** -- distributes destruction more evenly
4. **UNet with skip connections** -- preserves fine-grained info through the bottleneck
5. **Time embedding** -- sinusoidal encoding gives smooth noise-level awareness
6. **DDPM: 1000 steps, high quality** -- the original iterative approach
7. **DDIM: 50 steps, comparable quality** -- 20x faster, optionally deterministic
8. **Classifier-free guidance** -- trades diversity for quality by amplifying conditional signal
9. **2D distributions** -- perfect for building intuition before scaling to images